# Data Analyst Jobs in India: Exploratory Data Analysis (EDA)

**Data source:** Adzuna Jobs API (collected with Python, India)
**Tools used:** pandas, numpy, matplotlib, seaborn

## Aim of the project
Understand the **Data Analyst** job market in India from live job postings, so that a student can see what employers ask for.

## Questions we will answer
1. **Where** are data analyst jobs and **which companies** are hiring?
2. How much **experience** do employers ask for?
3. How much do data analyst jobs **pay**, and what changes the salary (experience, city, tools)?
4. Which **tools** (SQL, Excel, Power BI, Tableau, Python ...) are asked for most, and which tools come together?
5. Are there **remote** jobs, and how fresh are the postings?

## Steps
1. Load the data and select only data analyst jobs
2. Understand the data
3. Clean the data
4. Create new columns (feature engineering)
5. Analysis: location, companies, experience, time
6. Salary analysis
7. Tools and skills analysis
8. Final conclusions

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")   # hide warning messages to keep output clean

### Chart style (colours and helper functions)

Run this cell once. All charts below use the same colours, so the notebook looks like one report:
- **dark blue** = the bar we want you to notice, **light blue** = everything else
- **orange** = a highlight line (for example the median), **red** = negative values
- **grey** = "not mentioned" (missing information)

The plotting lines are the same ones from class (`plt.barh`, `plt.hist`, `sns.boxplot`, `sns.heatmap` ...).
The two small functions `tidy()` and `finish_chart()` only add the light grid lines, the title and the small note at the bottom.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import font_manager
from matplotlib.ticker import MaxNLocator

# ---- colours ----
SURFACE = "#fcfcfb"     # background
INK = "#0b0b0b"         # titles
INK2 = "#52514e"        # axis text and labels
GRID = "#e1e0d9"        # light grid lines
AXIS = "#c3c2b7"        # bottom line
BLUE = "#2a78d6"        # main colour
BLUE_LIGHT = "#86b6ef"  # "everything else" bars
ORANGE = "#eb6834"      # highlight line (median etc.)
RED = "#e34948"         # negative values
GREY = "#c3c2b7"        # "Not mentioned"

# one-colour map (light blue to dark blue) for heatmaps of counts or percentages
SEQ = LinearSegmentedColormap.from_list("seq", ["#cde2fb", "#6da7ec", "#2a78d6", "#184f95", "#0d366b"])
# blue - grey - red map for correlations (positive = blue, negative = red)
DIV = LinearSegmentedColormap.from_list("div", [RED, "#f0efec", BLUE])

# ---- font: use the first one that is installed on your computer ----
have = {f.name for f in font_manager.fontManager.ttflist}
font = next((f for f in ["Segoe UI", "Inter", "Arial", "DejaVu Sans"] if f in have), "sans-serif")

plt.rcParams.update({
    "font.family": font, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "savefig.dpi": 150,
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.edgecolor": AXIS, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlelocation": "left", "axes.titleweight": "bold", "axes.titlesize": 11.5,
    "xtick.major.size": 0, "ytick.major.size": 0, "legend.frameon": False,
})

def tidy(ax, grid="y"):
    """light grid lines and no left line. grid='x' for horizontal bars, grid='y' for columns / histograms"""
    ax.set_axisbelow(True)
    ax.grid(axis=grid, color=GRID, linewidth=0.8)
    ax.spines["bottom"].set_color(AXIS)
    ax.spines["left"].set_visible(False)

def finish_chart(title, subtitle=None, note=None):
    """big title and grey subtitle on top, small note at the bottom"""
    fig = plt.gcf()
    h = fig.get_figheight()
    fig.text(0.015, 1 - 0.30 / h, title, fontsize=15, fontweight="bold", ha="left", va="center")
    if subtitle:
        fig.text(0.015, 1 - 0.62 / h, subtitle, fontsize=10.5, color=INK2, ha="left", va="center")
    if note:
        fig.text(0.015, 0.20 / h, note, fontsize=8.5, color=INK2, ha="left", va="center")
    fig.tight_layout(rect=(0, 0.40 / h, 1, 1 - 0.85 / h))

def nice(labels):
    """'full_time' -> 'Full time', and put long labels on two lines"""
    out = []
    for l in labels:
        l = str(l).replace("_", " ")
        out.append((l[0].upper() + l[1:]).replace(" / ", " /\n"))
    return out

## 2. Load the data and select data analyst jobs

We have two files from the API: one with **all IT jobs** (18 different searches) and one with **data analyst jobs** only.
If you later collect more jobs with `src/fetch_data_analyst_jobs.py`, the notebook will also read `data/data_analyst_jobs_new.csv` automatically.
We join them, remove repeated jobs (same job id), and then keep only the rows where the **job title contains "data analyst"**.

In [ ]:
import pandas as pd
all_it = pd.read_csv("data/it_jobs_india.csv", encoding="utf-8-sig")
da_file = pd.read_csv("data/data_analyst_jobs.csv", encoding="utf-8-sig")

print("All IT jobs file     :", all_it.shape)
print("Data analyst file    :", da_file.shape)

In [ ]:
# The second file uses different column names, so rename them to match the first file
da_file = da_file.rename(columns={
    "id": "job_id",
    "company.display_name": "company",
    "location.display_name": "location",
    "category.label": "category",
    "redirect_url": "job_url",
})
da_file["search_term"] = "data analyst (second file)"

cols = list(all_it.columns)
frames = [all_it, da_file[cols]]

# Optional: extra jobs collected later with src/fetch_data_analyst_jobs.py
import os
new_path = "../data/data_analyst_jobs_new.csv"
if os.path.exists(new_path):
    new_file = pd.read_csv(new_path, encoding="utf-8-sig")
    print("Extra file found    :", new_file.shape)
    frames.append(new_file[cols])
else:
    print("No extra file found (that is fine)")

combined = pd.concat(frames, ignore_index=True)

print("Rows after joining              :", len(combined))
combined = combined.drop_duplicates(subset="job_id")
print("Rows after removing same job_id :", len(combined))

In [ ]:
# Keep only data analyst jobs: the title must contain "data analyst"
# (change the word here if you want to include other roles)
ROLE_WORD = "data analyst"

is_da = combined["title"].str.lower().str.contains(ROLE_WORD)
df = combined[is_da].copy().reset_index(drop=True)

print("Data analyst jobs selected :", len(df))
print("Other IT jobs left out     :", len(combined) - len(df))

In [ ]:
# Check that the filter picked the right titles
df["title"].str.lower().value_counts().head(10)

In [ ]:
# Related roles that we left out on purpose (you can include them by changing ROLE_WORD)
others = combined[~is_da]["title"].str.lower()
print("Business Intelligence / BI analyst titles left out :", others.str.contains("business intelligence|bi analyst").sum())
print("Business Analyst titles left out                   :", others.str.contains("business analyst").sum())

## 3. Understand the data

In [ ]:
df.head()

In [ ]:
print("Shape   :", df.shape)
df.info()

In [ ]:
# Missing values: count and percentage
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({"missing": missing, "percent": missing_pct})

In [ ]:
print("Fully duplicate rows :", df.duplicated().sum())
print("Same title+company+location+description :",
      df.duplicated(subset=["title", "company", "location", "description"]).sum())
print()
print(df["category"].value_counts())


In [ ]:
df[["salary_min", "salary_max"]].describe()

### What we noticed
- This is a **small dataset** (a few hundred rows at most), so every result must be read carefully. We will always show how many jobs are behind a number.
- Salary is missing for more than half of the jobs, and there are wrong values (for example 0).
- `salary_is_predicted` has only one value, so it gives no information.
- A few jobs are listed under other categories (for example Sales or HR) but the title is still "data analyst", so we keep them.
- `contract_type` is missing for most jobs.
- Descriptions are cut at 500 characters by the API, so tools found in them are a **minimum** count.

## 4. Data cleaning

**Step 1: Remove repeated postings and an useless column**

In [ ]:
before = len(df)
df = df.drop_duplicates(subset=["title", "company", "location", "description"])
print("Rows before :", before)
print("Rows after  :", len(df))

# salary_is_predicted has only one value
#df = df.drop(columns=["salary_is_predicted"])

**Step 2: Fix the date column**

In [ ]:
df["created"] = pd.to_datetime(df["created"])
df["year"] = df["created"].dt.year
df["month"] = df["created"].dt.month

# how old is each posting (compared with the newest posting in the data)
df["days_old"] = (df["created"].max() - df["created"]).dt.days

print("Oldest posting :", df["created"].min().date())
print("Newest posting :", df["created"].max().date())

**Step 3: Clean the location.** `location` looks like `"Pune, Maharashtra"`. We keep only the city. When it only says `"India"`, the city is not known.

In [ ]:
df["city"] = df["location"].apply(lambda x: x.split(",")[0].strip())
df["city"] = np.where(df["city"] == "India", "Not specified", df["city"])

# same city written in two ways
df["city"] = df["city"].replace({"Bengaluru": "Bangalore", "Gurgaon": "Gurugram"})

df["city"].value_counts().head(12)

**Step 4: Fill missing labels** (so they do not disappear from charts)

In [ ]:
df["company"] = df["company"].fillna("Not specified")
df["contract_type"] = df["contract_type"].fillna("Not mentioned")
df["contract_time"] = df["contract_time"].fillna("Not mentioned")

print(df["contract_type"].value_counts())
print(df["contract_time"].value_counts())

**Step 5: Clean the salary columns.** Salary in Adzuna India is yearly in rupees. Some values are wrong (0, 6) and some are clearly monthly salaries (like 20,000 to 25,000).
Rule: a salary is kept only if **both** min and max are at least ₹1,00,000 per year, otherwise it is treated as missing.
We also create `salary_lpa` (lakhs per annum), because that is how salaries are talked about in India.

In [ ]:
valid = (df["salary_min"] >= 100000) & (df["salary_max"] >= 100000) & (df["salary_max"] >= df["salary_min"])

print("Jobs with some salary value :", df["salary_max"].notna().sum())
print("Jobs with a usable salary   :", valid.sum())

df["salary_avg"] = np.where(valid, (df["salary_min"] + df["salary_max"]) / 2, np.nan)
df["salary_lpa"] = (df["salary_avg"] / 100000).round(2)

df["salary_lpa"].describe()

## 5. Create new columns (feature engineering)

**Seniority from the job title** (and the word "fresher" in the description).

In [ ]:
def get_level(row):
    title = str(row["title"]).lower()
    desc = str(row["description"]).lower()
    if "senior" in title or "sr." in title or "sr " in title or "lead" in title or "principal" in title or "staff" in title or "manager" in title:
        return "Senior / Lead"
    elif "junior" in title or "jr" in title or "intern" in title or "trainee" in title or "fresher" in title or "fresher" in desc or "entry level" in desc:
        return "Junior / Fresher"
    else:
        return "Not mentioned"

df["level"] = df.apply(get_level, axis=1)
df["level"].value_counts()

**Minimum years of experience** taken from the description with a pattern like "3+ years" or "2-5 years". Only the first number is kept. Numbers above 15 are wrong matches, so we remove them.

In [ ]:
df["min_exp"] = df["description"].str.extract(r"(\d{1,2})\s*(?:\+|-|to)?\s*(?:\d{1,2})?\s*\+?\s*(?:years|year|yrs|yr)", flags=2)[0]
df["min_exp"] = pd.to_numeric(df["min_exp"])
df["min_exp"] = df["min_exp"].where(df["min_exp"] <= 15)

print("Jobs where experience was found:", df["min_exp"].notna().sum(), "out of", len(df))
df["min_exp"].describe()

**Experience band**: group the minimum experience into three bands so that we can compare them.

In [ ]:
df["exp_band"] = df["min_exp"].apply(
    lambda x: "Not mentioned" if pd.isna(x) else "0-2 years" if x <= 2 else "3-5 years" if x <= 5 else "6+ years"
)
df["exp_band"].value_counts()

**Remote / hybrid** flag from the title, location and description.

In [ ]:
text = (df["title"].fillna("") + " " + df["location"].fillna("") + " " + df["description"].fillna("")).str.lower()
df["remote"] = np.where(text.str.contains("remote|work from home|wfh|hybrid"), "Remote / Hybrid mentioned", "Not mentioned")
df["remote"].value_counts()

**Tools and skills.** We search the title and description for tools that data analysts use. Each tool gets a column with 1 (mentioned) or 0 (not mentioned).
Remember: descriptions are cut at 500 characters, so these are minimum counts.

In [ ]:
tools = {
    "SQL": r"\bsql\b|mysql|postgres",
    "Excel": r"\bexcel\b|vlookup|pivot table",
    "Power BI": r"power\s?-?bi",
    "Tableau": r"\btableau\b",
    "Python": r"\bpython\b|pandas",
    "R": r"r programming|r language|rstudio|python/r|python or r|python, r\b",
    "Looker": r"\blooker\b",
    "BigQuery": r"big\s?query",
    "Snowflake": r"\bsnowflake\b",
    "Azure": r"\bazure\b",
    "AWS": r"\baws\b|amazon web services",
    "GCP": r"\bgcp\b|google cloud",
    "Spark": r"\bspark\b|pyspark",
    "ETL": r"\betl\b",
    "SAS": r"\bsas\b",
    "SPSS": r"\bspss\b",
}

text = (df["title"].fillna("") + " " + df["description"].fillna("")).str.lower()

for name in tools:
    df["tool_" + name] = text.str.contains(tools[name], regex=True).astype(int)

tool_cols = [c for c in df.columns if c.startswith("tool_")]
df["tools_count"] = df[tool_cols].sum(axis=1)

print("Tool columns created :", len(tool_cols))
df[["title", "tools_count"] + tool_cols[:5]].head()

**Concepts and soft skills** (not tools, but words that show what the work is about).

In [ ]:
concepts = {
    "Dashboards": r"dashboard",
    "Reporting": r"\breport",
    "Data visualization": r"visuali[sz]ation",
    "Statistics": r"statistic",
    "Data cleaning / quality": r"data cleaning|data quality|data validation|clean(ing)? data",
    "Stakeholders": r"stakeholder",
    "Communication": r"communication",
    "Machine Learning": r"machine learning|\bml\b",
    "Insights / trends": r"insight|trend",
}

# remove phrases like "reporting to the manager", so they are not counted as reporting work
text = text.str.replace("reporting to", "", regex=False).str.replace("reports to", "", regex=False).str.replace("report to", "", regex=False)

for name in concepts:
    df["concept_" + name] = text.str.contains(concepts[name], regex=True).astype(int)

concept_cols = [c for c in df.columns if c.startswith("concept_")]
print("Concept columns created :", len(concept_cols))

**Save the cleaned data** so we can reuse it.

In [ ]:
df.to_csv("data/data_analyst_jobs_clean.csv", index=False, encoding="utf-8-sig")
print("Clean data shape:", df.shape)

## 6. Analysis: location, companies, experience and time

> **Important note:** the dataset is small, so a difference of a few jobs is not a real trend. We focus on big differences and always look at the counts.

### 6.1 Where are the jobs?

In [ ]:
known = df[df["city"] != "Not specified"]
city_counts = known["city"].value_counts().head(10)
top_share = round(city_counts.iloc[0] / len(known) * 100)

# top city in dark blue, the others in light blue
colors = [BLUE if i == 0 else BLUE_LIGHT for i in range(len(city_counts))]

plt.figure(figsize=(9, 6))
bars = plt.barh(city_counts.index, city_counts.values, color=colors, height=0.65)
plt.gca().invert_yaxis()                            # biggest bar on top
plt.gca().bar_label(bars, padding=4, color=INK2)    # number at the end of each bar
plt.xlabel("Number of jobs")
tidy(plt.gca(), grid="x")
finish_chart(f"{city_counts.index[0]} has {top_share}% of the data analyst jobs that name a city",
             "Top 10 cities by number of jobs",
             f"Based on {len(known)} jobs with a city. Noida, Gurugram and Delhi are counted separately.")
plt.savefig("outputs/figures/01_jobs_by_city.png")
plt.show()

print(city_counts)
print("\nJobs with no city (only 'India'):", (df["city"] == "Not specified").sum(),
      "->", round((df["city"] == "Not specified").mean() * 100, 1), "%")
print("Top 3 cities share of known-city jobs:",
      round(city_counts.head(3).sum() / len(known) * 100, 1), "%")

**Insight:**
- Bangalore has 53 jobs, about **42%** of the 126 jobs that show a city. Pune is second (13) and Noida third (9).
- Delhi NCR is split into Noida (9), Gurugram (6) and Delhi (4). Added together (19) it is bigger than Pune.
- Hyderabad and Mumbai have only 6 jobs each in this data.
- About 22% of the jobs (35) only say "India", so their city is unknown.

### 6.2 Which companies are hiring?

In [ ]:
named = df[df["company"] != "Not specified"]
company_counts = named["company"].value_counts().head(10)

plt.figure(figsize=(9, 6))
bars = plt.barh(company_counts.index, company_counts.values, color=BLUE, height=0.65)
plt.gca().invert_yaxis()
plt.gca().bar_label(bars, padding=4, color=INK2)
plt.xlabel("Number of jobs")
plt.xlim(0, company_counts.max() * 1.15)
tidy(plt.gca(), grid="x")
finish_chart(f"No company has more than {company_counts.max()} data analyst jobs",
             "Top 10 companies by number of jobs",
             f"{named['company'].nunique()} different companies for {len(named)} jobs. Some are recruitment platforms hiring for clients.")
plt.savefig("outputs/figures/02_top_companies.png")
plt.show()

print(company_counts)
print("\nUnique companies:", named["company"].nunique(), "for", len(named), "jobs")

**Insight:** Hiring is very spread out: 137 different companies for 159 jobs. The top company, Wissen Technology, has only 4 jobs, followed by Qrata and Multi Recruit with 3 each. So no single company dominates. Several of the names look like recruitment or hiring platforms (for example Weekday AI, whose posting says the role is for one of its clients), so the company shown is not always the final employer.

### 6.3 Contract time, contract type and seniority

In [ ]:
ct = df["contract_time"].value_counts()
cty = df["contract_type"].value_counts()
lv = df["level"].value_counts()

plt.figure(figsize=(14, 5.5))
panels = [(ct, "Contract time"), (cty, "Contract type"), (lv, "Seniority (from title)")]

for i, (counts, name) in enumerate(panels):
    plt.subplot(1, 3, i + 1)
    labels = nice(counts.index)
    colors = [GREY if l == "Not mentioned" else BLUE for l in labels]   # grey = missing
    bars = plt.bar(labels, counts.values, color=colors, width=0.6)
    plt.gca().bar_label(bars, padding=3, color=INK2)
    plt.ylim(0, counts.max() * 1.15)
    plt.title(name, pad=10)
    tidy(plt.gca(), grid="y")

finish_chart("Contract type and seniority are missing from almost every posting",
             "Number of jobs by what the posting says (grey = not mentioned)",
             "Seniority is taken from the job title only, so it is under-counted.")
plt.savefig("../outputs/figures/03_contract_and_level.png")
plt.show()

print(df["contract_time"].value_counts())
print(df["contract_type"].value_counts())
print(df["level"].value_counts())

**Insight:**
- 119 jobs (74%) are **full time**, and the other 26% do not say. No job is part time.
- Contract type is missing for 93% of jobs. Of the 11 that say, 7 are permanent and 4 are contract.
- Only 5 titles show a level (4 senior, 1 junior). About 97% are just "Data Analyst", so we cannot study seniority from titles. The experience in years (next chart) is more useful.

### 6.4 How much experience is asked?

In [ ]:
exp = df["min_exp"].dropna()
band_order = ["0-2 years", "3-5 years", "6+ years", "Not mentioned"]
bc = df["exp_band"].value_counts().reindex(band_order)

plt.figure(figsize=(14, 5.5))

plt.subplot(1, 2, 1)
plt.hist(exp, bins=np.arange(-0.5, 15.5, 1), color=BLUE, edgecolor=SURFACE, linewidth=1.5)   # one bar per whole year
plt.ylim(0, plt.ylim()[1] * 1.25)      # extra space on top for the label
plt.axvline(exp.median(), color=ORANGE, linewidth=2)
plt.text(exp.median() + 0.25, plt.ylim()[1] * 0.93, f"Median: {exp.median():.0f} years", color=INK2, fontsize=10)
plt.xticks(range(0, 16, 2))
plt.xlabel("Minimum years of experience asked")
plt.ylabel("Number of jobs")
plt.title("Jobs that state their experience", pad=10)
tidy(plt.gca(), grid="y")

plt.subplot(1, 2, 2)
colors = [GREY if b == "Not mentioned" else BLUE for b in bc.index]
bars = plt.bar(bc.index, bc.values, color=colors, width=0.6)
plt.gca().bar_label(bars, padding=3, color=INK2)
plt.ylim(0, bc.max() * 1.12)
plt.title("All jobs, by experience band", pad=10)
tidy(plt.gca(), grid="y")

finish_chart(f"When stated, jobs ask for a median of {exp.median():.0f} years of experience",
             f"Only {len(exp)} of {len(df)} jobs state how much experience they want",
             "Experience was found by searching the description for phrases like '3+ years', so it is a minimum.")
plt.savefig("outputs/figures/04_experience.png")
plt.show()

print("Jobs where experience was found:", len(exp), "out of", len(df))
print("Median minimum experience      :", exp.median(), "years")
print(bc)

**Insight:** Experience could be found for only 35 of the 161 jobs (22%). Among those, the median minimum is **2 years**: 18 jobs ask for 0 to 2 years, 14 ask for 3 to 5 years and only 3 ask for 6 or more (the highest is 8). So when experience is stated, data analyst jobs lean towards entry and junior level. Be careful: the jobs that state experience may be different from the jobs that do not.

### 6.5 When were the jobs posted? Remote jobs

In [ ]:
year_counts = df["year"].value_counts().sort_index()
share_new = round((df["year"] == df["year"].max()).mean() * 100)
rm = df["remote"].value_counts().reindex(["Remote / Hybrid mentioned", "Not mentioned"])

plt.figure(figsize=(14, 5.5))

plt.subplot(1, 2, 1)
colors = [BLUE if y == year_counts.index.max() else BLUE_LIGHT for y in year_counts.index]   # newest year dark
bars = plt.bar(year_counts.index.astype(str), year_counts.values, color=colors, width=0.65)
plt.gca().bar_label(bars, padding=3, color=INK2)
plt.ylim(0, year_counts.max() * 1.12)
plt.xlabel("Year posted")
plt.title("Jobs by year posted", pad=10)
tidy(plt.gca(), grid="y")

plt.subplot(1, 2, 2)
bars = plt.bar(nice(rm.index), rm.values, color=[BLUE, GREY], width=0.5)
plt.gca().bar_label(bars, padding=3, color=INK2)
plt.ylim(0, rm.max() * 1.12)
plt.title("Remote / hybrid mentioned", pad=10)
tidy(plt.gca(), grid="y")

finish_chart(f"{share_new}% of the jobs were posted in {year_counts.index.max()}, and remote work is rarely mentioned",
             "Number of jobs",
             "Older postings may be listings that were never removed. Remote is searched in the title, location and description.")
plt.savefig("../outputs/figures/05_year_and_remote.png")
plt.show()

print(year_counts)
print("\nPosted in the last 30 days :", (df["days_old"] <= 30).sum(),
      "->", round((df["days_old"] <= 30).mean() * 100, 1), "%")
print("Remote / hybrid jobs       :", (df["remote"] != "Not mentioned").sum())

**Insight:**
- 95 jobs (59%) were posted in 2026 and 53 (33%) in the last 30 days, so most of the data is fresh.
- 35 jobs (22%) are from 2019 to 2022. These are probably old listings that were never removed.
- Only 7 jobs (about 4%) mention remote, work from home or hybrid. The real share may be a little higher because descriptions are cut at 500 characters.

### 6.5 When were the jobs posted? Remote jobs

## 7. Salary analysis

In [ ]:
sal = df[df["salary_lpa"].notna()].copy()
print("Jobs with usable salary :", len(sal), "out of", len(df),
      "->", round(len(sal) / len(df) * 100, 1), "%")
sal["salary_lpa"].describe()

### 7.1 Salary distribution

In [ ]:
med = sal["salary_lpa"].median()
q1 = sal["salary_lpa"].quantile(0.25)
q3 = sal["salary_lpa"].quantile(0.75)

plt.figure(figsize=(9, 5.5))
plt.hist(sal["salary_lpa"], bins=np.arange(0, 35, 2.5), color=BLUE, edgecolor=SURFACE, linewidth=1.5)
plt.ylim(0, plt.ylim()[1] * 1.25)      # extra space on top for the label
plt.axvline(med, color=ORANGE, linewidth=2)
plt.text(med + 0.5, plt.ylim()[1] * 0.93, f"Median: {med} LPA", color=INK2, fontsize=10)
plt.xlabel("Average salary (LPA)")
plt.ylabel("Number of jobs")
plt.gca().yaxis.set_major_locator(MaxNLocator(integer=True))   # whole numbers only (you cannot have half a job)
tidy(plt.gca(), grid="y")
finish_chart(f"A typical listed salary for a data analyst is {med} LPA",
             f"The middle half of jobs pay between {q1:.0f} and {q3:.0f} LPA",
             f"Based on {len(sal)} jobs with a usable salary (yearly salary in lakhs of rupees).")
plt.savefig("outputs/figures/06_salary_hist.png")
plt.show()

print("Median :", sal["salary_lpa"].median(), "LPA")
print("Mean   :", round(sal["salary_lpa"].mean(), 2), "LPA")
print("Middle half of jobs pay between", sal["salary_lpa"].quantile(0.25), "and", sal["salary_lpa"].quantile(0.75), "LPA")

**Insight:** Only 57 of the 161 jobs (35%) have a usable salary. The **median is 7.5 LPA** and the middle half of the jobs pay between 6 and 11 LPA. The mean (9.7 LPA) is higher because a few jobs pay a lot (the highest is 32.4 LPA), so the median describes a typical job better.

### 7.2 Salary and experience
Only a few jobs have **both** a salary and an experience value, so we plot the points one by one instead of drawing a trend line.

In [ ]:
both = sal[sal["min_exp"].notna()]
top = both.loc[both["salary_lpa"].idxmax()]

plt.figure(figsize=(9, 5.5))
plt.scatter(both["min_exp"], both["salary_lpa"], color=BLUE, s=90, edgecolor=SURFACE, linewidth=1.5, alpha=0.9)
plt.annotate(f"One job: {top['min_exp']:.0f} years,\n{top['salary_lpa']:.0f} LPA",
             xy=(top["min_exp"], top["salary_lpa"]), xytext=(top["min_exp"] - 2.3, top["salary_lpa"] - 1),
             color=INK2, fontsize=10, ha="center",
             arrowprops={"arrowstyle": "-", "color": AXIS})
plt.xlabel("Minimum years of experience asked")
plt.ylabel("Average salary (LPA)")
tidy(plt.gca(), grid="both")
finish_chart("There are too few jobs to link salary with experience",
             f"Only {len(both)} of {len(df)} jobs state both a salary and the experience",
             "Each dot is one job. Dots on top of each other are jobs with the same values.")
plt.savefig("../outputs/figures/07_salary_vs_experience.png")
plt.show()

print("Jobs with both salary and experience:", len(both), "out of", len(df))
print(both.groupby("exp_band")["salary_lpa"].agg(["count", "median"]).round(1))

### 7.3 Salary by city
Only Bangalore has enough jobs with salary (21). Every other city has 3 or fewer. So we compare three groups: **Bangalore**, **other known cities** and **jobs where the city is not given**.

In [ ]:
sal["city_group"] = sal["city"].apply(
    lambda x: "Bangalore" if x == "Bangalore" else "City not given" if x == "Not specified" else "Other cities"
)
order = ["Bangalore", "Other cities", "City not given"]
counts = sal["city_group"].value_counts()
med = sal.groupby("city_group")["salary_lpa"].median()

plt.figure(figsize=(9, 5.5))
sns.boxplot(data=sal, x="city_group", y="salary_lpa", order=order, hue="city_group",
            palette={"Bangalore": BLUE, "Other cities": BLUE_LIGHT, "City not given": BLUE_LIGHT},
            legend=False, saturation=1, width=0.5, linewidth=1.2, linecolor=INK2,
            medianprops={"color": INK, "linewidth": 2.5},
            flierprops={"marker": "o", "markersize": 5, "markerfacecolor": SURFACE, "markeredgecolor": INK2})
for i, g in enumerate(order):
    plt.text(i + 0.30, med[g], f"{med[g]:.1f}", va="center", color=INK2, fontsize=10)   # median value
plt.xticks(range(3), [f"{g}\n(n = {counts[g]})" for g in order])
plt.xlabel("")
plt.ylabel("Average salary (LPA)")
tidy(plt.gca(), grid="y")
finish_chart(f"Bangalore's median salary is {med['Bangalore']:.1f} LPA vs {med['Other cities']:.1f} in other cities",
             "Salary by city group. The line inside each box is the median.",
             "Groups are small (18 to 21 jobs), so treat the gap as a hint, not a proven difference.")
plt.savefig("outputs/figures/08_salary_by_city.png")
plt.show()

sal.groupby("city_group")["salary_lpa"].agg(["count", "median", "mean"]).round(1).reindex(order)

**Insight:** Bangalore has a median of **9.5 LPA** (21 jobs), compared with 7.2 LPA for other cities (18 jobs) and 7.0 LPA where the city is not given (18 jobs). So Bangalore looks about 2 to 2.5 LPA higher, but the boxes overlap a lot and the groups are small, so treat it as a hint, not a proven difference. Cities like Delhi, Mumbai and Noida have 3 or fewer salaries each, so they cannot be compared one by one. Also, 60% of Bangalore jobs have no salary at all.

## 8. Tools and skills analysis

### 8.1 Which tools are asked for most?

In [ ]:
tool_pct = (df[tool_cols].mean() * 100).sort_values(ascending=False)
tool_pct.index = [t.replace("tool_", "") for t in tool_pct.index]

none_pct = round((df["tools_count"] == 0).mean() * 100)
colors = [BLUE if i == 0 else BLUE_LIGHT for i in range(len(tool_pct))]   # most asked tool dark

plt.figure(figsize=(9, 7))
bars = plt.barh(tool_pct.index, tool_pct.values, color=colors, height=0.65)
plt.gca().invert_yaxis()
plt.gca().bar_label(bars, labels=[f"{v:.1f}%" for v in tool_pct.values], padding=4, color=INK2)
plt.xlim(0, tool_pct.max() * 1.12)
plt.xlabel("% of data analyst jobs mentioning the tool")
tidy(plt.gca(), grid="x")
finish_chart(f"{tool_pct.index[0]} is the most asked tool: {tool_pct.iloc[0]:.0f}% of jobs",
             "Share of jobs whose title or description mentions each tool",
             f"{none_pct}% of jobs name none of these tools. Descriptions are cut at 500 characters, so these are minimum values.")
plt.savefig("outputs/figures/09_top_tools.png")
plt.show()

print(tool_pct.round(1))
print("\nJobs mentioning no tool from our list:", (df["tools_count"] == 0).sum(),
      "->", round((df["tools_count"] == 0).mean() * 100, 1), "%")
print("Median number of tools per job (jobs with at least one):", df[df["tools_count"] > 0]["tools_count"].median())

**Insight:**
- **SQL** is the most asked tool (42 jobs, 26%), then Python (16%), Power BI and Excel (about 11% each) and Tableau (7.5%). A BI tool (Power BI, Tableau or Looker) appears in 21 jobs (13%).
- Everything else (ETL, Spark, Looker, BigQuery, R, cloud, SAS, SPSS) is at 5% or less.
- 64% of the jobs name none of our 16 tools. The reason is mainly the 500-character cut and generic job posts. So read these numbers as **minimums**. Excel in particular is probably asked more than shown.

### 8.2 Concepts and soft skills

In [ ]:
concept_pct = (df[concept_cols].mean() * 100).sort_values(ascending=False)
concept_pct.index = [c.replace("concept_", "") for c in concept_pct.index]

colors = [BLUE if i < 2 else BLUE_LIGHT for i in range(len(concept_pct))]   # top two dark

plt.figure(figsize=(9, 5.5))
bars = plt.barh(concept_pct.index, concept_pct.values, color=colors, height=0.65)
plt.gca().invert_yaxis()
plt.gca().bar_label(bars, labels=[f"{v:.1f}%" for v in concept_pct.values], padding=4, color=INK2)
plt.xlim(0, concept_pct.max() * 1.12)
plt.xlabel("% of data analyst jobs mentioning it")
tidy(plt.gca(), grid="x")
finish_chart(f"Machine learning appears in only {concept_pct['Machine Learning']:.0f}% of data analyst jobs",
             "Share of jobs whose title or description mentions each word",
             "Descriptions are cut at 500 characters, so these are minimum values.")
plt.savefig("outputs/figures/10_concepts.png")
plt.show()

print(concept_pct.round(1))

**Insight:** The words that appear most are about finding insights and trends (32%) and reporting (29%), then dashboards (15%), stakeholders (14%), data cleaning / quality (13%), visualization (12%), statistics and communication (11% each). Machine Learning appears in only 3% of the jobs. So in this data a data analyst job is mostly about cleaning data, building reports and dashboards, and explaining insights to the business, not about machine learning.

### 8.3 Which tools come together? (the typical analyst "stack")

In [ ]:
top_tools = ["tool_" + t for t in tool_pct.head(7).index]
corr = df[top_tools].corr()
corr.columns = [c.replace("tool_", "") for c in corr.columns]
corr.index = corr.columns

sub = corr.iloc[1:, :-1]                                   # drop the empty first row / last column
mask = np.triu(np.ones_like(sub, dtype=bool), k=1)         # hide the mirrored half
pairs = corr.where(~np.eye(len(corr), dtype=bool)).stack()
best = pairs.idxmax()

plt.figure(figsize=(8, 6.5))
sns.heatmap(sub, mask=mask, annot=True, fmt=".2f", cmap=DIV, vmin=-1, vmax=1, center=0,
            linewidths=2, linecolor=SURFACE, square=True, cbar_kws={"shrink": 0.7},
            annot_kws={"fontsize": 10})
plt.gca().tick_params(length=0)
plt.yticks(rotation=0)
finish_chart(f"{best[0]} and {best[1]} are the pair most often asked together",
             "Correlation between tools across jobs (1 = always together, 0 = unrelated, below 0 = rarely together)",
             "Only the 7 most asked tools are shown. Based on tools found in titles and descriptions.")
plt.savefig("outputs/figures/11_tools_together.png")
plt.show()

In [ ]:
# The classic combination: SQL + Excel + a BI tool (Power BI / Tableau / Looker)
bi_tool = (df["tool_Power BI"] == 1) | (df["tool_Tableau"] == 1) | (df["tool_Looker"] == 1)

print("Jobs asking SQL                      :", (df["tool_SQL"] == 1).sum())
print("Jobs asking Excel                    :", (df["tool_Excel"] == 1).sum())
print("Jobs asking a BI tool                :", bi_tool.sum())
print("Jobs asking SQL + Excel              :", ((df["tool_SQL"] == 1) & (df["tool_Excel"] == 1)).sum())
print("Jobs asking SQL + a BI tool          :", ((df["tool_SQL"] == 1) & bi_tool).sum())
print("Jobs asking SQL + Python             :", ((df["tool_SQL"] == 1) & (df["tool_Python"] == 1)).sum())
print("Jobs asking SQL + Excel + a BI tool  :", ((df["tool_SQL"] == 1) & (df["tool_Excel"] == 1) & bi_tool).sum())

**Insight:**
- **SQL and Python** are the strongest pair (correlation 0.66): 24 jobs ask for both, which is 57% of all SQL jobs.
- **Power BI and Tableau** also go together (0.52), and 8 jobs ask for both.
- SQL goes with Excel (0.39, 13 jobs) and SQL with a BI tool appears in 14 jobs.
- The classic combination SQL + Excel + a BI tool appears in only 6 jobs.
- ETL and Spark are only weakly linked with the other tools. They show up in a different kind of job, closer to data engineering.

In this sample, SQL is the core and employers add either Python or a BI tool to it.

### 8.4 Do tools raise salary?
For each popular tool, we compare the **median salary** of jobs that mention it with jobs that do not. This shows a link, not proof that the tool *causes* higher pay. We only use tools with at least 8 salary values.

In [ ]:
rows = []
for t in tool_pct.head(10).index:
    col = "tool_" + t
    with_t = sal[sal[col] == 1]["salary_lpa"]
    without = sal[sal[col] == 0]["salary_lpa"]
    if len(with_t) >= 8:
        rows.append({"tool": t, "salary_jobs_with_tool": len(with_t),
                     "median_with": with_t.median(), "median_without": without.median()})

tool_salary = pd.DataFrame(rows)
tool_salary["difference"] = (tool_salary["median_with"] - tool_salary["median_without"]).round(2)
tool_salary = tool_salary.sort_values("difference", ascending=False)

max_gap = tool_salary["difference"].abs().max()
colors = [BLUE if d >= 0 else RED for d in tool_salary["difference"]]    # blue = pays more, red = pays less

plt.figure(figsize=(9, 5))
bars = plt.barh(tool_salary["tool"], tool_salary["difference"], color=colors, height=0.5)
plt.gca().invert_yaxis()
plt.gca().bar_label(bars, labels=[f"{d:+.1f}" for d in tool_salary["difference"]], padding=4, color=INK2)
plt.axvline(0, color=INK2, linewidth=1)
plt.xlim(-5, 5)      # same scale as other salary gaps, so small gaps look small
plt.xlabel("Median salary difference (LPA)   <  jobs with the tool pay less  |  pay more  >")
tidy(plt.gca(), grid="x")
finish_chart(f"Salary gaps between tools are small: at most {max_gap:.0f} LPA",
             "Median salary of jobs that mention the tool minus jobs that do not",
             "Only tools with at least 8 salaries are shown. A link is not proof that the tool causes higher pay.")
plt.savefig("outputs/figures/12_tool_salary_gap.png")
plt.show()

tool_salary.round(2)

**Insight:** Jobs that name SQL have a median salary of 8.5 LPA (20 jobs) against 7.5 LPA for the others, a gap of +1.0. Python gives +0.5 (11 jobs) and Excel +0.0 (8 jobs). Power BI and Tableau are left out because they have fewer than 8 salaries each. All the gaps are small and the groups are small, so we **cannot say** that any tool raises pay in this sample.

### 8.5 Correlation heatmap

In [ ]:
corr2 = sal[["salary_lpa", "tools_count", "days_old"]].corr()

labels = {"salary_lpa": "Salary (LPA)", "tools_count": "Number of tools", "days_old": "Days since posted"}
c = corr2.rename(index=labels, columns=labels)
sub = c.iloc[1:, :-1]
mask = np.triu(np.ones_like(sub, dtype=bool), k=1)

plt.figure(figsize=(8, 4.8))
sns.heatmap(sub, mask=mask, annot=True, fmt=".2f", cmap=DIV, vmin=-1, vmax=1, center=0,
            linewidths=2, linecolor=SURFACE, square=True, cbar_kws={"shrink": 0.8},
            annot_kws={"fontsize": 11})
plt.gca().tick_params(length=0)
plt.yticks(rotation=0)
finish_chart("Salary has no strong link with the number of tools or the posting age",
             "Correlation (1 = move together, 0 = unrelated, below 0 = move opposite)",
             f"Based on {len(sal)} jobs with a salary. With so few jobs, small values can be chance.")
plt.savefig("outputs/figures/13_correlation.png")
plt.show()

corr2.round(2)

## 9. Final conclusions

### What we found (161 data analyst jobs)
1. **Location:** Bangalore has about 42% of the jobs that show a city. Pune is second and Delhi NCR (Noida + Gurugram + Delhi) is also big when added together.
2. **Companies:** Hiring is very spread out (137 companies for 159 jobs). Some postings come from recruitment platforms.
3. **Experience:** where experience is stated, the median is **2 years**, and most jobs ask for 0 to 5 years. Data analyst is an entry-friendly role in this data.
4. **Salary:** the typical (median) listed salary is **7.5 LPA**, and the middle half of jobs pay 6 to 11 LPA. Bangalore looks about 2 LPA higher, but the sample is small.
5. **Tools:** SQL is the most asked tool (26%), then Python (16%), Power BI and Excel (about 11%) and Tableau (8%). SQL + Python is the strongest pair, and Power BI + Tableau also go together.
6. **The work itself:** insights, reporting, dashboards and stakeholders appear much more than Machine Learning (3%).
7. **Other points:** about 4% of jobs mention remote or hybrid work, 59% were posted in 2026, and 33% in the last 30 days.

### What this means for a beginner (based on this sample only)
Start with **SQL**, then add **Python (pandas)** and **one BI tool** (Power BI or Tableau). Practise building dashboards and explaining insights, because those words appear in many job posts.

### Limitations (please read)
- The dataset is **small**: 161 jobs, and only 57 have a usable salary. Many comparisons use groups of 3 to 20 jobs, so differences can be due to chance.
- Our "data analyst" search stopped at 150 jobs (3 pages), so it is **not the complete** market.
- Descriptions are cut at **500 characters**, so tools, experience and remote work are under-counted. Experience was found for only 22% of the jobs.
- Related roles are not included: 65 Business Intelligence / BI analyst jobs and 9 Business Analyst jobs were left out on purpose.
- 22% of the jobs are from 2019 to 2022 and may be old listings.
- Tools and skills were found by searching words, which is simple and can make mistakes.

### Next steps
- Run `src/fetch_data_analyst_jobs.py` to collect more data analyst jobs (more pages and more search words), then run this notebook again. It will read the new file automatically.
- Change `ROLE_WORD` to `"data analyst|business intelligence|business analyst"` to include the related roles and compare them.
- With more data, test salary against experience, city and tools with a simple linear regression.